In [ ]:
import time
import requests
import pandas as pd

# =========================
# CONFIGURATIONS
# =========================
API_KEY = "API"  # your API key
URL = "https://maps.googleapis.com/maps/api/place/textsearch/json"
LANG = "pt-BR" #you can change the language
PATH_CSV = "path"

# Terms you tested on google maps
terms = [
    "tabacaria",
    "narguilé",
    "smoke shop",
    "hookah bar",
    "vape shop",
    "cigar shop",
    "cigar lounge",
]

# =========================
# READ CSV (A=city, B=UF)
# =========================
df_cidades = pd.read_csv(PATH_CSV, dtype=str)
# Detect probably columns or assume two first
city_cols = [c for c in df_cidades.columns if c.strip().lower() in {"cidade", "municipio", "município"}]
uf_cols   = [c for c in df_cidades.columns if c.strip().lower() in {"uf", "estado", "sigla_uf"}]

if not city_cols or not uf_cols:
    # Use the first two columns if the names do not match
    first_two = df_cidades.columns[:2].tolist()
    df_cidades = df_cidades.rename(columns={first_two[0]: "cidade", first_two[1]: "uf"})
else:
    df_cidades = df_cidades.rename(columns={city_cols[0]: "cidade", uf_cols[0]: "uf"})

df_cidades["cidade"] = df_cidades["cidade"].astype(str).str.strip()
df_cidades["uf"] = df_cidades["uf"].astype(str).str.strip().str.upper()
df_cidades = df_cidades.dropna(subset=["cidade", "uf"])

# =========================
# FUNCTION RETRY/BACKOFF
# =========================
def call_text_search(params, sleep_after=1.0, max_retries=5):
    retries = 0
    while True:
        resp = requests.get(URL, params=params, timeout=30)
        data = resp.json()
        status = data.get("status", "UNKNOWN_ERROR")

        if status == "OK":
            time.sleep(sleep_after)
            return data

        # INVALID_REQUEST it can occur immediately after requesting the next page
        if status == "INVALID_REQUEST" and "pagetoken" in params:
            time.sleep(2.2)
            retries += 1
            if retries <= max_retries:
                continue

        if status in {"OVER_QUERY_LIMIT", "UNKNOWN_ERROR"}:
            wait = min(2**retries, 60)  # backoff 60s
            time.sleep(wait)
            retries += 1
            if retries <= max_retries:
                continue

        # ZERO_RESULTS, REQUEST_DENIED etc: returns as is
        return data

# =========================
# COLLECT
# =========================
rows = []
for _, r in df_cidades.iterrows():
    cidade, uf = r["cidade"], r["uf"]
    local_str = f"{cidade} - {uf}, Brasil"

    for termo in terms:
        query = f"{termo} em {local_str}"
        params = {"query": query, "language": LANG, "key": API_KEY}
        print(f"Buscando: {query}")

        data = call_text_search(params)
        status = data.get("status")
        if status != "OK":
            print(f"  Status: {status} - {data.get('error_message', 'sem erro detalhado')}")
            continue

        while True:
            for place in data.get("results", []):
                loc = (place.get("geometry") or {}).get("location") or {}
                rows.append({
                    "termo": termo,
                    "cidade": cidade,
                    "uf": uf,
                    "query": query,
                    "place_id": place.get("place_id"),
                    "nome": place.get("name"),
                    "endereco": place.get("formatted_address"),
                    "latitude": loc.get("lat"),
                    "longitude": loc.get("lng"),
                    "nota": place.get("rating"),
                    "qtd_avaliacoes": place.get("user_ratings_total"),
                    "status_comercial": place.get("business_status"),
                    "tipo_primario": (place.get("types") or [None])[0] if place.get("types") else None,
                })

            next_token = data.get("next_page_token")
            if not next_token:
                break

            time.sleep(2.2)
            data = call_text_search({"pagetoken": next_token, "language": LANG, "key": API_KEY})

# =========================
# DEDUP + SAVE
# =========================
df = pd.DataFrame(rows)
if not df.empty:
    # Prioritizes the record with the most reviews for each place_id
    df = df.sort_values(["place_id", "qtd_avaliacoes"], ascending=[True, False])
    df = df.drop_duplicates(subset=["place_id"], keep="first")

    out = "tabacarias_brasil_total.csv"
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"{len(df)} lugares únicos salvos em {out}")
else:
    print("Nenhum resultado coletado.")


In [ ]:
# -*- coding: utf-8 -*-
# Compare tobacco shops and schools by distance (meters) and categorize as <50m, <100m, and <200m
# Requirements: pandas, numpy. (Optional: scikit-learn; fallback: scipy; final fallback: block calculation)

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

# ======= INPUT FILES =======
TABACARIAS_CSV = "tabacarias_brasil_total.csv"
# try this name first; if it doesn't exist, it autodetects further down
ESCOLAS_CSV    = "geocodificacao_escolas_completas.csv"

# ======= OUTPUT FILES =======
OUT_DIR = Path("resultados_dist")
OUT_DIR.mkdir(exist_ok=True)
OUT_MATCHES = OUT_DIR / "tabacarias_escolas_distancias.csv"
OUT_50  = OUT_DIR / "pares_lt50m.csv"
OUT_100 = OUT_DIR / "pares_lt100m.csv"
OUT_200 = OUT_DIR / "pares_lt200m.csv"

# ---------- utils ----------
def _norm_colnames(df):
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _load_csv_any(path, **kwargs):
    # try the automatic separator, then ';'.
    try:
        return pd.read_csv(path, dtype=str, sep=None, engine="python", **kwargs)
    except Exception:
        return pd.read_csv(path, dtype=str, sep=";", **kwargs)

def _coerce_float(s):
    return pd.to_numeric(s, errors="coerce")

# haversine (meters) — inputs in RADIANS
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

# ---------- loading and normalization ----------
def load_tabacarias(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"File not found: {path}")
    df = _load_csv_any(path)
    df = _norm_colnames(df)
    lat_col = _pick_col(df, ["latitude", "lat", "y"])
    lon_col = _pick_col(df, ["longitude", "lon", "lng", "x"])
    if not lat_col or not lon_col:
        raise ValueError("Not found lat/long in TABACARIAS.")

    keep_cols = [lat_col, lon_col]
    for extra in ["uf", "cidade", "nome", "place_id", "endereco", "termo"]:
        if extra in df.columns:
            keep_cols.append(extra)
    df = df[keep_cols].copy()

    df["lat"] = _coerce_float(df[lat_col])
    df["lon"] = _coerce_float(df[lon_col])
    df = df.dropna(subset=["lat","lon"])
    df = df[(df["lat"].between(-90,90)) & (df["lon"].between(-180,180))].reset_index(drop=True)
    return df

def load_escolas(path):
    path = _autodetect_escolas_path(path)
    df = _load_csv_any(path)
    df = _norm_colnames(df)
    # colunas esperadas do consolidado
    lat_col = _pick_col(df, ["lat","latitude","y"])
    lon_col = _pick_col(df, ["lon","longitude","x"])
    if not lat_col or not lon_col:
        raise ValueError("Não encontrei colunas de lat/lon em ESCOLAS.")
    keep_cols = [lat_col, lon_col]
    for extra in ["co_entidade","no_entidade","sg_uf","no_municipio","precisao"]:
        if extra in df.columns:
            keep_cols.append(extra)
    df = df[keep_cols].copy()

    df["lat"] = _coerce_float(df[lat_col])
    df["lon"] = _coerce_float(df[lon_col])
    df = df.dropna(subset=["lat","lon"])
    df = df[(df["lat"].between(-90,90)) & (df["lon"].between(-180,180))].reset_index(drop=True)
    return df

# ---------- NN wrappers ----------
# Plan A: Fast spatial indexing using scikit-learn's BallTree to find the nearest neighbor in milliseconds
def nearest_with_sklearn(tab, esc):
    from sklearn.neighbors import BallTree
    X = np.radians(tab[["lat","lon"]].to_numpy())
    Y = np.radians(esc[["lat","lon"]].to_numpy())
    tree = BallTree(Y, metric="haversine")
    dist_rad, idx = tree.query(X, k=1)
    dist_m = dist_rad.flatten() * 6371000.0
    return idx.flatten(), dist_m

# Plan B: Fallback using SciPy's cKDTree for spatial approximation if scikit-learn is missing
def nearest_with_scipy(tab, esc):
    from scipy.spatial import cKDTree
    lat_scale = 111_320.0
    esc_rad_lat = np.radians(esc["lat"].to_numpy())
    cos_lat = np.cos(esc_rad_lat).mean()
    esc_xy = np.column_stack([
        esc["lon"].to_numpy() * (lat_scale * cos_lat),
        esc["lat"].to_numpy() * lat_scale
    ])
    tab_xy = np.column_stack([
        tab["lon"].to_numpy() * (lat_scale * cos_lat),
        tab["lat"].to_numpy() * lat_scale
    ])
    tree = cKDTree(esc_xy)
    dist_m, idx = tree.query(tab_xy, k=1)
    # refina com haversine
    t_lat = np.radians(tab["lat"].to_numpy())
    t_lon = np.radians(tab["lon"].to_numpy())
    e_lat = np.radians(esc["lat"].to_numpy()[idx])
    e_lon = np.radians(esc["lon"].to_numpy()[idx])
    dist_m_ref = haversine_m(t_lat, t_lon, e_lat, e_lon)
    return idx, dist_m_ref

# Plan C: Pure math brute-force fallback, processing data in chunks to prevent RAM overflow
def nearest_bruteforce_chunked(tab, esc, chunk=5000):
    t_lat = np.radians(tab["lat"].to_numpy())
    t_lon = np.radians(tab["lon"].to_numpy())
    e_lat = np.radians(esc["lat"].to_numpy())
    e_lon = np.radians(esc["lon"].to_numpy())
    n_t = t_lat.shape[0]
    best_idx = np.empty(n_t, dtype=int)
    best_dist = np.full(n_t, np.inf, dtype=float)
    for i in range(0, n_t, chunk):
        j = min(i+chunk, n_t)
        d = haversine_m(
            t_lat[i:j, None], t_lon[i:j, None],
            e_lat[None, :],    e_lon[None, :]
        )  # (blk, n_esc)
        idx_local = d.argmin(axis=1)
        dist_local = d[np.arange(j-i), idx_local]
        best_idx[i:j] = idx_local
        best_dist[i:j] = dist_local
    return best_idx, best_dist

def find_nn(tab, esc):
    try:
        return nearest_with_sklearn(tab, esc)
    except Exception:
        try:
            return nearest_with_scipy(tab, esc)
        except Exception:
            return nearest_bruteforce_chunked(tab, esc)

# Orchestrator: Tries Plan A, if it fails tries Plan B, and as a last resort runs Plan C
def match_nearest(tab, esc, use_uf_partition=True):
    tab = tab.copy()
    esc = esc.copy()
    if use_uf_partition and ("uf" in tab.columns) and ("sg_uf" in esc.columns):
        all_parts = []
        for uf, bloc in tab.groupby(tab["uf"].str.upper()):
            cand = esc[esc["sg_uf"].str.upper() == uf]
            if cand.empty:
                cand = esc
            idx, dist = find_nn(bloc, cand)
            matched = bloc.reset_index(drop=True).copy()
            matched["idx_escola"] = idx
            matched["dist_m"] = dist
            schools_mat = cand.reset_index(drop=True).iloc[idx].reset_index(drop=True)
            for c in ["co_entidade","no_entidade","sg_uf","no_municipio","lat","lon","precisao"]:
                if c in schools_mat.columns:
                    matched[f"escola_{c}"] = schools_mat[c].to_numpy()
            all_parts.append(matched)
        return pd.concat(all_parts, ignore_index=True)
    else:
        idx, dist = find_nn(tab, esc)
        matched = tab.reset_index(drop=True).copy()
        matched["idx_escola"] = idx
        matched["dist_m"] = dist
        schools_mat = esc.reset_index(drop=True).iloc[idx].reset_index(drop=True)
        for c in ["co_entidade","no_entidade","sg_uf","no_municipio","lat","lon","precisao"]:
            if c in schools_mat.columns:
                matched[f"escola_{c}"] = schools_mat[c].to_numpy()
        return matched

def main():
    print("reading tabacarias…")
    tab = load_tabacarias(TABACARIAS_CSV)
    print(f"Tabacarias with coordinates: {len(tab):,}")

    print("reading escolas…")
    esc_path = _autodetect_escolas_path(ESCOLAS_CSV)
    print(f"Using escolas file: {esc_path}")
    esc = load_escolas(esc_path)
    print(f"Escolas with coordinates: {len(esc):,}")

    # NM match
    print("Calculating nearest neighbor…")
    matched = match_nearest(tab, esc, use_uf_partition=True)

    # band flags
    matched["dist_<50m"]  = matched["dist_m"] < 50
    matched["dist_<100m"] = matched["dist_m"] < 100
    matched["dist_<200m"] = matched["dist_m"] < 200

    # range label (mutually exclusive)
    bins = np.where(matched["dist_<50m"], "<50m",
            np.where(matched["dist_<100m"], "<100m",
            np.where(matched["dist_<200m"], "<200m", ">200m")))
    matched["band"] = bins

    # save
    cols_first = ["nome","endereco","cidade","uf","lat","lon","dist_m","band","dist_<50m","dist_<100m","dist_<200m"]
    cols_first = [c for c in cols_first if c in matched.columns]
    all_cols = cols_first + [c for c in matched.columns if c not in cols_first]
    matched[all_cols].to_csv(OUT_MATCHES, index=False, encoding="utf-8")
    print(f"Arquivo salvo: {OUT_MATCHES} ({len(matched):,} linhas)")

    # distance cutoff
    matched.loc[matched["dist_<50m"]].to_csv(OUT_50, index=False, encoding="utf-8")
    matched.loc[matched["dist_<100m"]].to_csv(OUT_100, index=False, encoding="utf-8")
    matched.loc[matched["dist_<200m"]].to_csv(OUT_200, index=False, encoding="utf-8")
    print(f"Recortes salvos: {OUT_50}, {OUT_100}, {OUT_200}")

    # general summary
    print("\ngeneral summary:")
    total = len(matched)
    for lbl in ["<50m","<100m","<200m"]:
        n = (matched["band"] == lbl).sum()
        print(f"  {lbl}: {n:,} ({n/total:.1%})")
    n200 = (matched["band"] == ">200m").sum()
    print(f"  >200m: {n200:,} ({n200/total:.1%})")

    # ---- resumo por UF (robusto) ----
    cols = ["dist_<50m", "dist_<100m", "dist_<200m"]
    uf_col = None
    for cand in ["uf", "escola_sg_uf", "SG_UF"]:  # tente ambos esquemas
        if cand in matched.columns:
            uf_col = cand
            break
    if uf_col:
        for c in cols:
            matched[c] = matched[c].astype(int)
        resumo_uf = (
            matched
            .groupby(uf_col)[cols]
            .sum(numeric_only=True)
            .sort_values("dist_<200m", ascending=False)
        )
        print("\nTop 10 UFs por pares <200m:")
        print(resumo_uf.head(10))
    else:
        print("\n[AVISO] Nenhuma coluna de UF encontrada para resumo por UF.")

if __name__ == "__main__":
    main()

